# 01. Runtime 请求路径：HTTP 到 Scheduler 再回到用户

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## 入口层：`sglang serve` / `python -m sglang.launch_server`

启动入口会解析 `ServerArgs`，然后根据模式分流到 HTTP、gRPC、Ray、encoder-only 或 disaggregation server。默认 HTTP 模式进入 `sglang.srt.entrypoints.http_server.launch_server`。


In [ ]:
for path in ["python/sglang/launch_server.py", "python/sglang/srt/entrypoints/http_server.py"]:
    print("\n==", path)
    for row in find_defs(path, names={"run_server", "launch_server", "generate_request"}):
        print(row)


In [ ]:
show_lines("python/sglang/launch_server.py", 12, 45)


### 启动分流：`run_server` 只决定拓扑，不做推理

```python
# python/sglang/launch_server.py:12-43
  12: suppress_noisy_warnings()
      # 注：函数/库调用：`suppress_noisy_warnings` 把当前阶段委托给外部 helper 或库实现。
  14: 
  15: def run_server(server_args):
      # 注：函数定义：`run_server` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  16:     """Run the server based on server_args.grpc_mode and server_args.encoder_only."""
  17:     if server_args.encoder_only:
      # 注：分支判断：根据启动参数 `server_args.encoder_only` 选择服务拓扑、通信方式或功能开关。
  18:         # For encoder disaggregation
  19:         if server_args.grpc_mode:
      # 注：分支判断：根据启动参数 `server_args.grpc_mode` 选择服务拓扑、通信方式或功能开关。
  20:             from sglang.srt.disaggregation.encode_grpc_server import (
  21:                 serve_grpc_encoder,
  22:             )
  23: 
  24:             asyncio.run(serve_grpc_encoder(server_args))
      # 注：对象/库方法调用：`asyncio.run` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
  25:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
  26:             from sglang.srt.disaggregation.encode_server import launch_server
  27: 
  28:             launch_server(server_args)
      # 注：函数/库调用：`launch_server` 把当前阶段委托给外部 helper 或库实现。
  29:     elif server_args.grpc_mode:
      # 注：补充分支：前面的启动模式不匹配时，再按 `server_args.grpc_mode` 选择另一种服务拓扑。
  30:         # TODO: Once the native Rust gRPC server starts alongside HTTP in the
  31:         # default path below (controlled by SGLANG_ENABLE_GRPC / SGLANG_GRPC_PORT),
  32:         # remove this legacy SMG path and the grpc_mode flag.
  33:         from sglang.srt.entrypoints.grpc_server import serve_grpc
  34: 
  35:         asyncio.run(serve_grpc(server_args))
      # 注：对象/库方法调用：`asyncio.run` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
  36:     elif server_args.use_ray:
      # 注：补充分支：前面的启动模式不匹配时，再按 `server_args.use_ray` 选择另一种服务拓扑。
  37:         # Ray mode: HTTP mode with Ray backend.
  38:         try:
      # 注：异常边界：下面的调用可能跨进程、I/O 或用户输入，失败时需要清理内部状态。
  39:             from sglang.srt.ray.http_server import launch_server
  40:         except ImportError:
      # 注：异常处理分支：把失败转换成可控清理、缓存失败对象或用户可见错误。
  41:             raise ImportError(
      # 注：函数/库调用：`ImportError` 把当前阶段委托给外部 helper 或库实现。
  42:                 "Ray is required for --use-ray mode. "
  43:                 "Install it with: pip install 'sglang[ray]'"
```

**读这段时抓住：**

- `encoder_only`、`grpc_mode`、`use_ray` 是互斥的服务形态分支；默认路径才进入 HTTP server。
- 这里没有 tokenizer、scheduler、model runner 的细节，说明启动入口的职责是选择 runtime 拓扑。
- 以后排查“为什么我的参数没有进入默认 HTTP server”时，先看这段分流条件。


### 默认 HTTP 路径的核心：`http_server.launch_server`

`run_server` 只是把默认分支交给这里；真正把 HTTP server、TokenizerManager、Scheduler 子进程、
Detokenizer 子进程装配起来的是 `launch_server`。下面这段代码很短，但它是默认服务模式的总装入口。

```python
# python/sglang/srt/entrypoints/http_server.py:2455-2497
# 阶段 1：声明可替换的构造函数。测试、Ray/fork、私有部署可以替换这些函数。

def launch_server(
    server_args: ServerArgs,
    # 注：server_args 是整个服务的配置对象；后续 engine 子进程和 HTTP server 都共享这份配置。
    init_tokenizer_manager_func: Callable = init_tokenizer_manager,
    # 注：可替换 TokenizerManager 初始化函数，测试或私有 fork 可以在这里注入自定义 manager。
    run_scheduler_process_func: Callable = run_scheduler_process,
    # 注：可替换 Scheduler 子进程入口；调度器和模型加载都在这条路径里。
    run_detokenizer_process_func: Callable = run_detokenizer_process,
    # 注：可替换 Detokenizer 子进程入口；输出 token 到文本的转换由它负责。
    execute_warmup_func: Callable = _execute_server_warmup,
    # 注：warmup 在 HTTP server lifecycle 中执行，用于提前触发模型/图/缓存准备。
    launch_callback: Optional[Callable[[], None]] = None,
    # 注：服务完成启动后可调用的回调，常用于测试或嵌入式部署通知外部系统。
):
    """
    Launch SRT (SGLang Runtime) Server.

    The SRT server consists of an HTTP server and an SRT engine.
    ...
    """
    # 阶段 2：先启动 SRT engine 的内部组件。
    # TokenizerManager 在主进程；Scheduler/Detokenizer 通常在子进程。
    (
        tokenizer_manager,
        template_manager,
        port_args,
        scheduler_init_result,
        subprocess_watchdog,
    ) = Engine._launch_subprocesses(...)
    # 注：返回的 tokenizer_manager 留在主进程；scheduler_init_result 携带 scheduler 回传的能力信息。

    # 阶段 3：再把 FastAPI/uvicorn 绑定到这些组件上，开始接 HTTP 请求。
    _setup_and_run_http_server(
        server_args,
        # 注：HTTP endpoint 会通过 global state 间接调用 tokenizer_manager。
        tokenizer_manager,
        template_manager,
        port_args,
        scheduler_init_result.scheduler_infos,
        # 注：scheduler_infos 让 HTTP/tokenizer 侧知道 max input length、model info 等运行时事实。
        subprocess_watchdog,
        execute_warmup_func=execute_warmup_func,
        launch_callback=launch_callback,
    )
```

**读这段时抓住：**

- `launch_server` 是默认 HTTP 服务的“装配函数”，不是请求处理函数。
- 它先启动 engine 子系统，再启动 HTTP server；这保证 API 开始监听前 scheduler/model 已经在加载或就绪流程中。
- 函数参数都是可注入的 callable，这也是测试和私有 fork 可以替换启动行为的原因。
- `scheduler_init_result.scheduler_infos` 会被传给 HTTP 层，后续 API 层才能知道 max input length 等 scheduler 能力。


### `Engine._launch_subprocesses` 上半段：准备环境、端口和 Scheduler

```python
# python/sglang/srt/entrypoints/engine.py:751-844
 751:     def _launch_subprocesses(
      # 注：函数定义：`_launch_subprocesses` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 752:         cls,
 753:         server_args: ServerArgs,
 754:         init_tokenizer_manager_func: Callable,
 755:         run_scheduler_process_func: Callable,
 756:         run_detokenizer_process_func: Callable,
 757:         port_args: Optional[PortArgs] = None,
 758:     ) -> Tuple[
 759:         TokenizerManager,
 760:         TemplateManager,
 761:         PortArgs,
 762:         SchedulerInitResult,
 763:         Optional[SubprocessWatchdog],
 764:     ]:
 765:         """Launch the TokenizerManager in the main process, the Scheduler in a subprocess, and the DetokenizerManager in another subprocess.
 766: 
 767:         Returns:
 768:             Tuple of (tokenizer_manager, template_manager, port_args, scheduler_init_result, subprocess_watchdog).
 769:         """
 770:         # Configure global environment
 771:         configure_logger(server_args)
      # 注：函数/库调用：`configure_logger` 把当前阶段委托给外部 helper 或库实现。
 772:         _set_envs_and_config(server_args)
      # 注：函数/库调用：`_set_envs_and_config` 把当前阶段委托给外部 helper 或库实现。
 773: 
 774:         # Defensive: ensure plugins loaded (may already be loaded by
 775:         # Engine.__init__ or CLI entry).
 776:         load_plugins()
      # 注：关键调用：`load_plugins` - 加载插件，让插件有机会注册模型、参数 hook 或 speculative/grammar 扩展。
 777: 
 778:         server_args.check_server_args()
      # 注：关键调用：`server_args.check_server_args` - 在进程启动前做参数一致性校验，避免子进程启动后才失败。
 779:         _set_gc(server_args)
      # 注：函数/库调用：`_set_gc` 把当前阶段委托给外部 helper 或库实现。
 780: 
 781:         # Allocate ports for inter-process communications
 782:         if port_args is None:
      # 注：分支判断：只有满足 `port_args is None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 783:             port_args = PortArgs.init_new(server_args)
      # 注：局部状态绑定：`port_args` 保存配置快照，下面的分支通常会基于它选择执行路径。
 784:         logger.info(f"{server_args=}")
      # 注：对象/库方法调用：`logger.info` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 785: 
 786:         # Start the engine info bootstrap server if per-rank info is needed.
 787:         engine_info_bootstrap_server = None
      # 注：局部状态绑定：`engine_info_bootstrap_server` 保存本阶段的中间结果，后续几行通常会立即消费它。
 788:         if (
      # 注：多行分支开始：完整条件在接下来几行，通常用于组合多个请求参数或运行时状态。
 789:             server_args.remote_instance_weight_loader_start_seed_via_transfer_engine
 790:             and server_args.node_rank == 0
 791:         ):
 792:             bootstrap_port = server_args.engine_info_bootstrap_port
      # 注：局部状态绑定：`bootstrap_port` 保存通信端点，后续 manager 间 IPC 会使用它。
 793:             if not is_port_available(bootstrap_port):
      # 注：分支判断：只有满足 `not is_port_available(bootstrap_port)` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 794:                 raise RuntimeError(
      # 注：函数/库调用：`RuntimeError` 把当前阶段委托给外部 helper 或库实现。
 795:                     f"engine_info_bootstrap_port {bootstrap_port} is already in use. "
 796:                     f"When running multiple instances on the same node, each instance must use a "
 797:                     f"different --engine-info-bootstrap-port."
 798:                 )
 799:             engine_info_bootstrap_server = EngineInfoBootstrapServer(
      # 注：局部状态绑定：`engine_info_bootstrap_server` 保存本阶段的中间结果，后续几行通常会立即消费它。
 800:                 host=server_args.host, port=bootstrap_port
      # 注：局部状态绑定：`host` 保存本阶段的中间结果，后续几行通常会立即消费它。
 801:             )
 802: 
 803:         if (
      # 注：多行分支开始：完整条件在接下来几行，通常用于组合多个请求参数或运行时状态。
 804:             server_args.reasoning_parser == "auto"
 805:             or server_args.tool_call_parser == "auto"
 806:         ):
 807:             resolve_auto_parsers(server_args)
      # 注：函数/库调用：`resolve_auto_parsers` 把当前阶段委托给外部 helper 或库实现。
 808: 
 809:         # Launch scheduler processes
 810:         scheduler_init_result, scheduler_procs = cls._launch_scheduler_processes(
      # 注：局部状态绑定：`scheduler_init_result, scheduler_procs` 保存本阶段的中间结果，后续几行通常会立即消费它。
 811:             server_args, port_args, run_scheduler_process_func
 812:         )
 813:         scheduler_init_result.engine_info_bootstrap_server = (
      # 注：局部状态绑定：`scheduler_init_result.engine_info_bootstrap_server` 保存本阶段的中间结果，后续几行通常会立即消费它。
 814:             engine_info_bootstrap_server
 815:         )
 816: 
 817:         if (
      # 注：多行分支开始：完整条件在接下来几行，通常用于组合多个请求参数或运行时状态。
 818:             server_args.enable_elastic_expert_backup
 819:             and server_args.elastic_ep_backend is not None
 820:         ):
 821:             run_expert_backup_manager(server_args, port_args)
      # 注：函数/库调用：`run_expert_backup_manager` 把当前阶段委托给外部 helper 或库实现。
 822: 
 823:         if server_args.node_rank >= 1:
      # 注：分支判断：根据启动参数 `server_args.node_rank >= 1` 选择服务拓扑、通信方式或功能开关。
 824:             # In multi-node cases, non-zero rank nodes do not need to run tokenizer or detokenizer,
 825:             # so they can just wait here.
 826:             scheduler_init_result.wait_for_ready()
      # 注：关键调用：`scheduler_init_result.wait_for_ready` - 等待 scheduler/model worker 完成初始化并回传可服务状态。
 827: 
 828:             if os.getenv("SGLANG_BLOCK_NONZERO_RANK_CHILDREN") == "0":
      # 注：分支判断：只有满足 `os.getenv("SGLANG_BLOCK_NONZERO_RANK_CHILDREN") == "0"` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 829:                 # When using `Engine` as a Python API, we don't want to block here.
 830:                 return (
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 831:                     None,
 832:                     None,
 833:                     port_args,
 834:                     scheduler_init_result,
 835:                     None,
 836:                 )
 837: 
 838:             launch_dummy_health_check_server(
      # 注：函数/库调用：`launch_dummy_health_check_server` 把当前阶段委托给外部 helper 或库实现。
 839:                 server_args.host, server_args.port, server_args.enable_metrics
 840:             )
 841: 
 842:             scheduler_init_result.wait_for_completion()
      # 注：对象/库方法调用：`scheduler_init_result.wait_for_completion` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 843:             return (
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 844:                 None,
```

**读这段时抓住：**

- `configure_logger`、`_set_envs_and_config`、`server_args.check_server_args()` 是进程启动前的全局准备。
- `PortArgs.init_new` 分配 manager 间 IPC 地址；后面的 Tokenizer/Scheduler/Detokenizer 都靠它通信。
- `_launch_scheduler_processes` 先启动 scheduler，因为模型加载和 scheduler 能力信息都来自那里。
- 多节点非 0 rank 可能只跑 scheduler/worker，不跑 tokenizer/detokenizer；这解释了为什么 HTTP API 通常只在 rank0 暴露。


### `Engine._launch_subprocesses` 下半段：Detokenizer、TokenizerManager、ready 信号和 watchdog

```python
# python/sglang/srt/entrypoints/engine.py:845-891
 845:                 None,
 846:                 port_args,
 847:                 scheduler_init_result,
 848:                 None,
 849:             )
 850: 
 851:         # Launch detokenizer process(es) — optionally fronted by a router when
 852:         # detokenizer_worker_num > 1.
 853:         detoken_procs, detoken_names = cls._launch_detokenizer_subprocesses(
      # 注：局部状态绑定：`detoken_procs, detoken_names` 保存本阶段的中间结果，后续几行通常会立即消费它。
 854:             server_args=server_args,
 855:             port_args=port_args,
 856:             run_detokenizer_process_func=run_detokenizer_process_func,
 857:         )
 858:         for p in detoken_procs:
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
 859:             scheduler_init_result.all_child_pids.append(p.pid)
      # 注：对象/库方法调用：`scheduler_init_result.all_child_pids.append` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 860: 
 861:         # Init tokenizer manager first, as the bootstrap server is initialized here
 862:         if server_args.tokenizer_worker_num == 1:
      # 注：分支判断：根据启动参数 `server_args.tokenizer_worker_num == 1` 选择服务拓扑、通信方式或功能开关。
 863:             tokenizer_manager, template_manager = init_tokenizer_manager_func(
      # 注：局部状态绑定：`tokenizer_manager, template_manager` 保存本阶段的中间结果，后续几行通常会立即消费它。
 864:                 server_args, port_args
 865:             )
 866:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 867:             # Launch multi-tokenizer router
 868:             tokenizer_manager = MultiTokenizerRouter(server_args, port_args)
      # 注：局部状态绑定：`tokenizer_manager` 保存本阶段的中间结果，后续几行通常会立即消费它。
 869:             template_manager = None
      # 注：局部状态绑定：`template_manager` 保存本阶段的中间结果，后续几行通常会立即消费它。
 870: 
 871:         # Wait for the model to finish loading
 872:         scheduler_init_result.wait_for_ready()
      # 注：关键调用：`scheduler_init_result.wait_for_ready` - 等待 scheduler/model worker 完成初始化并回传可服务状态。
 873: 
 874:         # Get back some info from scheduler to tokenizer_manager
 875:         tokenizer_manager.max_req_input_len = scheduler_init_result.scheduler_infos[0][
      # 注：局部状态绑定：`tokenizer_manager.max_req_input_len` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 876:             "max_req_input_len"
 877:         ]
 878: 
 879:         # Set up subprocess liveness watchdog to detect crashes
 880:         # Note: RayEngine returns scheduler_procs=None as it uses Ray actors instead of mp.Process
 881:         processes = list(scheduler_procs or [])
      # 注：局部状态绑定：`processes` 保存本阶段的中间结果，后续几行通常会立即消费它。
 882:         names = [f"scheduler_{i}" for i in range(len(processes))]
      # 注：局部状态绑定：`names` 保存本阶段的中间结果，后续几行通常会立即消费它。
 883:         processes.extend(detoken_procs)
      # 注：对象/库方法调用：`processes.extend` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 884:         names.extend(detoken_names)
      # 注：对象/库方法调用：`names.extend` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 885:         subprocess_watchdog = SubprocessWatchdog(
      # 注：局部状态绑定：`subprocess_watchdog` 保存本阶段的中间结果，后续几行通常会立即消费它。
 886:             processes=processes, process_names=names
      # 注：局部状态绑定：`processes` 保存本阶段的中间结果，后续几行通常会立即消费它。
 887:         )
 888:         subprocess_watchdog.start()
      # 注：对象/库方法调用：`subprocess_watchdog.start` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 889: 
 890:         return (
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 891:             tokenizer_manager,
```

**读这段时抓住：**

- detokenizer 子进程先启动，然后主进程初始化 TokenizerManager 或 MultiTokenizerRouter。
- `scheduler_init_result.wait_for_ready()` 是关键同步点：HTTP 层不能盲目认为模型已经可用。
- `max_req_input_len` 从 scheduler 回写给 tokenizer manager，输入长度校验才有真实模型/部署上下文。
- `SubprocessWatchdog` 把 scheduler/detokenizer 崩溃变成主进程可感知事件，是生产服务韧性的一部分。


### `_setup_and_run_http_server`：把 engine 状态挂到 FastAPI app 上

```python
# python/sglang/srt/entrypoints/http_server.py:2251-2335
2251: def _setup_and_run_http_server(
      # 注：函数定义：`_setup_and_run_http_server` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
2252:     server_args: ServerArgs,
2253:     tokenizer_manager,
2254:     template_manager,
2255:     port_args: PortArgs,
2256:     scheduler_infos: List[Dict],
2257:     subprocess_watchdog: Optional[SubprocessWatchdog],
2258:     execute_warmup_func: Callable = _execute_server_warmup,
2259:     launch_callback: Optional[Callable[[], None]] = None,
2260: ):
2261:     """Set up global state, configure middleware, and run uvicorn.
2262: 
2263:     Called by launch_server after subprocesses have been launched.
2264:     """
2265:     # Set global states
2266:     set_global_state(
      # 注：关键调用：`set_global_state` - 把 manager/template/scheduler 信息放入 HTTP 全局状态，endpoint 之后通过它拿到 runtime 对象。
2267:         _GlobalState(
      # 注：函数/库调用：`_GlobalState` 把当前阶段委托给外部 helper 或库实现。
2268:             tokenizer_manager=tokenizer_manager,
2269:             template_manager=template_manager,
2270:             scheduler_info=scheduler_infos[0],
2271:         )
2272:     )
2273: 
2274:     # Store watchdog on tokenizer_manager (single source of truth for SIGQUIT handler)
2275:     if tokenizer_manager is not None:
      # 注：分支判断：只有满足 `tokenizer_manager is not None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
2276:         tokenizer_manager._subprocess_watchdog = subprocess_watchdog
      # 注：局部状态绑定：`tokenizer_manager._subprocess_watchdog` 保存本阶段的中间结果，后续几行通常会立即消费它。
2277: 
2278:     if server_args.enable_metrics:
      # 注：分支判断：根据启动参数 `server_args.enable_metrics` 选择服务拓扑、通信方式或功能开关。
2279:         add_prometheus_track_response_middleware(app)
      # 注：函数/库调用：`add_prometheus_track_response_middleware` 把当前阶段委托给外部 helper 或库实现。
2280: 
2281:     # Pass additional arguments to the lifespan function.
2282:     # They will be used for additional initialization setups.
2283:     if server_args.tokenizer_worker_num == 1:
      # 注：分支判断：根据启动参数 `server_args.tokenizer_worker_num == 1` 选择服务拓扑、通信方式或功能开关。
2284:         # If it is single tokenizer mode, we can pass the arguments by attributes of the app object.
2285:         app.is_single_tokenizer_mode = True
      # 注：对象状态写入：`app.is_single_tokenizer_mode` 修改 `app` 的可见状态，读源码时要继续追踪它在哪里被消费。
2286:         app.server_args = server_args
      # 注：对象状态写入：`app.server_args` 修改 `app` 的可见状态，读源码时要继续追踪它在哪里被消费。
2287:         app.warmup_thread_kwargs = dict(
      # 注：对象状态写入：`app.warmup_thread_kwargs` 修改 `app` 的可见状态，读源码时要继续追踪它在哪里被消费。
2288:             server_args=server_args,
2289:             launch_callback=launch_callback,
2290:             execute_warmup_func=execute_warmup_func,
2291:         )
2292: 
2293:         # Add api key authorization
2294:         # This is only supported in single tokenizer mode.
2295:         #
2296:         # Backward compatibility:
2297:         # - api_key only: behavior matches legacy (all endpoints require api_key)
2298:         # - no keys: legacy had no restriction; ADMIN_FORCE endpoints must still be rejected when
2299:         #   admin_api_key is not configured.
2300:         if (
      # 注：多行分支开始：完整条件在接下来几行，通常用于组合多个请求参数或运行时状态。
2301:             server_args.api_key
2302:             or server_args.admin_api_key
2303:             or app_has_admin_force_endpoints(app)
      # 注：函数/库调用：`app_has_admin_force_endpoints` 把当前阶段委托给外部 helper 或库实现。
2304:         ):
2305:             from sglang.srt.utils.auth import add_api_key_middleware
2306: 
2307:             add_api_key_middleware(
      # 注：关键调用：`add_api_key_middleware` - 把 API key/admin API key 鉴权逻辑注册到 FastAPI middleware。
2308:                 app,
2309:                 api_key=server_args.api_key,
2310:                 admin_api_key=server_args.admin_api_key,
2311:             )
2312:     else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
2313:         # If it is multi-tokenizer mode, we need to write the arguments to shared memory
2314:         # for other worker processes to read.
2315:         app.is_single_tokenizer_mode = False
      # 注：对象状态写入：`app.is_single_tokenizer_mode` 修改 `app` 的可见状态，读源码时要继续追踪它在哪里被消费。
2316:         multi_tokenizer_args_shm = write_data_for_multi_tokenizer(
      # 注：局部状态绑定：`multi_tokenizer_args_shm` 保存配置快照，下面的分支通常会基于它选择执行路径。
2317:             port_args, server_args, scheduler_infos[0]
2318:         )
2319: 
2320:     try:
      # 注：异常边界：下面的调用可能跨进程、I/O 或用户输入，失败时需要清理内部状态。
2321:         # Update logging configs
2322:         set_uvicorn_logging_configs(server_args)
      # 注：函数/库调用：`set_uvicorn_logging_configs` 把当前阶段委托给外部 helper 或库实现。
2323: 
2324:         if server_args.ssl_certfile:
      # 注：分支判断：根据启动参数 `server_args.ssl_certfile` 选择服务拓扑、通信方式或功能开关。
2325:             logger.info(
      # 注：对象/库方法调用：`logger.info` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
2326:                 f"SSL enabled: certfile={server_args.ssl_certfile}, "
2327:                 f"keyfile={server_args.ssl_keyfile}"
2328:             )
2329: 
2330:         # Listen for HTTP requests
2331:         if server_args.tokenizer_worker_num == 1:
      # 注：分支判断：根据启动参数 `server_args.tokenizer_worker_num == 1` 选择服务拓扑、通信方式或功能开关。
2332:             if server_args.enable_http2:
      # 注：分支判断：根据启动参数 `server_args.enable_http2` 选择服务拓扑、通信方式或功能开关。
2333:                 logger.info(
      # 注：对象/库方法调用：`logger.info` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
2334:                     f"Starting embedded Granian HTTP/2 server on "
2335:                     f"{server_args.host}:{server_args.port}"
```

**读这段时抓住：**

- `set_global_state` 把 tokenizer_manager/template_manager/scheduler_info 暴露给 FastAPI endpoint 依赖函数。
- single-tokenizer 模式直接把 `server_args` 和 warmup kwargs 挂到 `app`；multi-tokenizer 模式则写共享内存给 worker 读。
- API key middleware 只在 single-tokenizer 模式直接添加；多 tokenizer/多 worker 的 HTTP 启动路径不同。
- 最后根据 HTTP/2、SSL refresh、tokenizer worker 数选择 Granian 或 uvicorn 的不同启动方式。


## Manager 边界

SGLang 的默认 HTTP 服务不是一个“大函数直接 forward”。它把工作拆成几类 manager：

- `TokenizerManager`：API 请求、chat template、多模态预处理、tokenization、streaming 状态。
- `Scheduler`：waiting/running queue、prefix cache、KV pool、grammar queue、batch formation、forward 调用。
- `DetokenizerManager`：把 token ids 增量转回文本，并负责流式输出中的 offset 管理。
- `TpModelWorker` / `ModelRunner`：真正持有模型权重、attention backend、CUDA graph runner 和 KV cache pool。

这条边界非常重要：多数 Feature 要么在 tokenizer 层增加请求字段，要么在 scheduler 层改变调度/缓存，要么在 model runner/attention backend 层改变 tensor 执行。


### `TokenizerManager.__init__` 的尾部：通信、dispatcher、采样参数类

```python
# python/sglang/srt/managers/tokenizer_manager.py:545-575
 545:             start_cpu_monitor_thread("tokenizer")
      # 注：函数/库调用：`start_cpu_monitor_thread` 把当前阶段委托给外部 helper 或库实现。
 546: 
 547:         if self.server_args.gc_warning_threshold_secs > 0.0:
      # 注：分支判断：根据启动参数 `self.server_args.gc_warning_threshold_secs > 0.0` 选择服务拓扑、通信方式或功能开关。
 548:             configure_gc_warning(self.server_args.gc_warning_threshold_secs)
      # 注：函数/库调用：`configure_gc_warning` 把当前阶段委托给外部 helper 或库实现。
 549:         self.soft_watchdog = Watchdog.create(
      # 注：成员变量写入：`self.soft_watchdog` 会留在对象生命周期中，后续方法可能依赖这份状态。
 550:             debug_name="TokenizerManager",
 551:             watchdog_timeout=self.server_args.soft_watchdog_timeout,
 552:             soft=True,
 553:             test_stuck_time=envs.SGLANG_TEST_STUCK_TOKENIZER.get(),
 554:         )
 555: 
 556:     def init_request_dispatcher(self):
      # 注：函数定义：`init_request_dispatcher` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 557:         self._result_dispatcher = TypeBasedDispatcher(
      # 注：成员变量写入：`self._result_dispatcher` 会留在对象生命周期中，后续方法可能依赖这份状态。
 558:             [
 559:                 (AbortReq, self._handle_abort_req),
 560:                 (OpenSessionReqOutput, self._handle_open_session_req_output),
 561:                 (
 562:                     UpdateWeightFromDiskReqOutput,
 563:                     self._handle_update_weights_from_disk_req_output,
 564:                 ),
 565:                 (FreezeGCReq, lambda x: None),
 566:                 # For handling case when scheduler skips detokenizer and forwards back to the tokenizer manager, we ignore it.
 567:                 (HealthCheckOutput, lambda x: None),
 568:                 (ActiveRanksOutput, self.update_active_ranks),
 569:             ]
 570:         )
 571:         self.init_communicators(self.server_args)
      # 注：关键调用：`self.init_communicators` - 初始化 tokenizer manager 与 scheduler/detokenizer 间的 IPC 通道。
 572: 
 573:         self.sampling_params_class = SamplingParams
      # 注：成员变量写入：`self.sampling_params_class` 会留在对象生命周期中，后续方法可能依赖这份状态。
 574:         self.signal_handler_class = SignalHandler
      # 注：成员变量写入：`self.signal_handler_class` 会留在对象生命周期中，后续方法可能依赖这份状态。
 575: 
```

**读这段时抓住：**

- `TypeBasedDispatcher` 让 tokenizer manager 能处理 scheduler/detokenizer 返回的多种对象，而不是写一串 endpoint 专用回调。
- `init_communicators` 建立 ZeroMQ/IPC 通道；这就是 API 进程和 scheduler/detokenizer 进程之间的边界。
- `sampling_params_class = SamplingParams` 意味着请求参数在进入 scheduler 前已经被规范化成内部采样对象。


### `DetokenizerManager.__init__`：它不是附属函数，而是独立进程角色

```python
# python/sglang/srt/managers/detokenizer_manager.py:89-133
  89: class DetokenizerManager(MultiHttpWorkerDetokenizerMixin):
      # 注：类定义：`DetokenizerManager` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
  90:     """DetokenizerManager is a process that detokenizes the token ids."""
  91: 
  92:     def __init__(
      # 注：函数定义：`__init__` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  93:         self,
  94:         server_args: ServerArgs,
  95:         port_args: PortArgs,
  96:     ):
  97:         # Init inter-process communication
  98:         self.init_ipc_channels(port_args, server_args)
      # 注：成员函数调用：`self.init_ipc_channels` 会读取或更新当前对象状态，建议继续查看该方法定义。
  99: 
 100:         # Init tokenizer
 101:         self.init_tokenizer(server_args)
      # 注：成员函数调用：`self.init_tokenizer` 会读取或更新当前对象状态，建议继续查看该方法定义。
 102: 
 103:         # Init running status
 104:         self.init_running_status(server_args)
      # 注：成员函数调用：`self.init_running_status` 会读取或更新当前对象状态，建议继续查看该方法定义。
 105: 
 106:         # Init dispatcher
 107:         self.init_request_dispatcher()
      # 注：成员函数调用：`self.init_request_dispatcher` 会读取或更新当前对象状态，建议继续查看该方法定义。
 108: 
 109:     def init_ipc_channels(self, port_args: PortArgs, server_args: ServerArgs):
      # 注：函数定义：`init_ipc_channels` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 110:         context = zmq.Context(2)
      # 注：局部状态绑定：`context` 保存本阶段的中间结果，后续几行通常会立即消费它。
 111:         self.recv_from_scheduler = get_zmq_socket(
      # 注：成员变量写入：`self.recv_from_scheduler` 会留在对象生命周期中，后续方法可能依赖这份状态。
 112:             context, zmq.PULL, port_args.detokenizer_ipc_name, True
 113:         )
 114:         # In multi-tokenizer mode, results are pushed back to each TokenizerWorker
 115:         # directly via SocketMapping inside multi_http_worker_event_loop, so the
 116:         # single send_to_tokenizer socket is unused.
 117:         if server_args.tokenizer_worker_num == 1:
      # 注：分支判断：根据启动参数 `server_args.tokenizer_worker_num == 1` 选择服务拓扑、通信方式或功能开关。
 118:             self.send_to_tokenizer = get_zmq_socket(
      # 注：成员变量写入：`self.send_to_tokenizer` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
 119:                 context, zmq.PUSH, port_args.tokenizer_ipc_name, False
 120:             )
 121: 
 122:     def init_tokenizer(self, server_args: ServerArgs):
      # 注：函数定义：`init_tokenizer` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 123:         if server_args.skip_tokenizer_init:
      # 注：分支判断：根据启动参数 `server_args.skip_tokenizer_init` 选择服务拓扑、通信方式或功能开关。
 124:             self.tokenizer = None
      # 注：成员变量写入：`self.tokenizer` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
 125:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 126:             self.tokenizer = get_tokenizer(
      # 注：成员变量写入：`self.tokenizer` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
 127:                 server_args.tokenizer_path,
 128:                 tokenizer_mode=server_args.tokenizer_mode,
 129:                 trust_remote_code=server_args.trust_remote_code,
 130:                 revision=server_args.revision,
 131:                 tokenizer_backend=server_args.tokenizer_backend,
 132:             )
 133: 
```

**读这段时抓住：**

- detokenizer 只从 scheduler 收 token id / batch output，再把文本结果发回 tokenizer manager。
- `skip_tokenizer_init` 会让它不加载 tokenizer；embedding/token-id-only 等路径可以绕开普通文本 detokenization。
- 这个边界减少 scheduler 的 CPU 文本处理负担，也让 streaming 输出可以和 GPU 调度解耦。


In [ ]:
paths = [
    "python/sglang/srt/entrypoints/engine.py",
    "python/sglang/srt/managers/tokenizer_manager.py",
    "python/sglang/srt/managers/scheduler.py",
    "python/sglang/srt/managers/detokenizer_manager.py",
]
for path in paths:
    print("\n==", path)
    for lineno, kind, name in find_defs(path):
        if name in {
            "init_tokenizer_manager", "run_scheduler_process", "run_detokenizer_process",
            "TokenizerManager", "generate_request", "Scheduler",
            "event_loop_normal", "event_loop_overlap", "run_batch", "process_batch_result",
            "DetokenizerManager",
        }:
            print(f"{lineno:4d} {kind:18s} {name}")


## 请求对象的生命周期

粗略流程：

1. FastAPI endpoint 接收 `/generate` 或 `/v1/chat/completions`。
2. OpenAI serving 层把 chat/completion 请求转为内部 `GenerateReqInput`。
3. `TokenizerManager.generate_request` 负责 tokenization、多模态输入处理、采样参数校验，并把 tokenized 请求发给 scheduler。
4. `Scheduler` 把请求放入 waiting queue；grammar 请求可能先进 `GrammarManager` 队列等编译完成。
5. `get_new_batch_prefill` 根据调度策略、KV 可用量、prefix hit、chunked prefill 等形成 `ScheduleBatch`。
6. `ForwardBatch.init_new` 将 CPU 侧调度数据变成 GPU 侧 tensor 元数据。
7. `ModelRunner.forward` 执行模型，attention 层通过 `RadixAttention` 转给当前 attention backend。
8. scheduler 处理 logits/sampling/finish reason，把结果发往 detokenizer/tokenizer manager。
9. API 层按 non-streaming 或 streaming 返回。


### `TokenizerManager.generate_request`：入站请求的主脊柱

```python
# python/sglang/srt/managers/tokenizer_manager.py:576-632
 576:     async def generate_request(
      # 注：异步函数定义：`generate_request` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 577:         self,
 578:         obj: Union[GenerateReqInput, EmbeddingReqInput],
 579:         request: Optional[fastapi.Request] = None,
 580:     ):
 581:         self.auto_create_handle_loop()
      # 注：成员函数调用：`self.auto_create_handle_loop` 会读取或更新当前对象状态，建议继续查看该方法定义。
 582: 
 583:         # Normalize the request
 584:         obj.normalize_batch_and_arguments()
      # 注：关键调用：`obj.normalize_batch_and_arguments` - 把单请求/批请求和参数别名规范化，后续路径才能统一处理。
 585:         self._set_default_priority(obj)
      # 注：关键调用：`self._set_default_priority` - 为请求补齐默认优先级，供 scheduler priority policy 使用。
 586: 
 587:         if isinstance(obj, GenerateReqInput) and obj.routed_dp_rank is not None:
      # 注：分支判断：只有满足 `isinstance(obj, GenerateReqInput) and obj.routed_dp_rank is not None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 588:             dp_size = self.server_args.dp_size
      # 注：局部状态绑定：`dp_size` 保存本阶段的中间结果，后续几行通常会立即消费它。
 589:             if dp_size <= 1 and obj.routed_dp_rank == 0:
      # 注：分支判断：只有满足 `dp_size <= 1 and obj.routed_dp_rank == 0` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 590:                 logger.debug(
      # 注：对象/库方法调用：`logger.debug` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 591:                     f"routed_dp_rank={obj.routed_dp_rank} is ignored because dp_size={dp_size}"
 592:                 )
 593:             elif obj.routed_dp_rank < 0 or obj.routed_dp_rank >= dp_size:
      # 注：补充分支：前面的条件不满足时，再检查 `obj.routed_dp_rank < 0 or obj.routed_dp_rank >= dp_size`。
 594:                 raise ValueError(
      # 注：函数/库调用：`ValueError` 把当前阶段委托给外部 helper 或库实现。
 595:                     f"routed_dp_rank={obj.routed_dp_rank} out of range [0, {dp_size})"
 596:                 )
 597: 
 598:         if self.server_args.tokenizer_worker_num > 1:
      # 注：分支判断：根据启动参数 `self.server_args.tokenizer_worker_num > 1` 选择服务拓扑、通信方式或功能开关。
 599:             self._attach_multi_http_worker_info(obj)
      # 注：成员函数调用：`self._attach_multi_http_worker_info` 会读取或更新当前对象状态，建议继续查看该方法定义。
 600:         self._init_req_state(obj, request)
      # 注：关键调用：`self._init_req_state` - 创建 rid 到 ReqState 的映射，后续 streaming/non-streaming 回包都靠它找到等待者。
 601:         try:
      # 注：异常边界：下面的调用可能跨进程、I/O 或用户输入，失败时需要清理内部状态。
 602:             if self.server_args.language_only:
      # 注：分支判断：根据启动参数 `self.server_args.language_only` 选择服务拓扑、通信方式或功能开关。
 603:                 self._handle_epd_disaggregation_encode_request(obj)
      # 注：成员函数调用：`self._handle_epd_disaggregation_encode_request` 会读取或更新当前对象状态，建议继续查看该方法定义。
 604: 
 605:             # Log the request
 606:             self.request_logger.log_received_request(obj, self.tokenizer, request)
      # 注：成员函数调用：`self.request_logger.log_received_request` 会读取或更新当前对象状态，建议继续查看该方法定义。
 607: 
 608:             async with self.is_pause_cond:
 609:                 await self.is_pause_cond.wait_for(lambda: not self.is_pause)
      # 注：成员函数调用：`self.is_pause_cond.wait_for` 会读取或更新当前对象状态，建议继续查看该方法定义。
 610: 
 611:             async with self.model_update_lock.reader_lock:
 612:                 await self._validate_and_resolve_lora(obj)
      # 注：成员函数调用：`self._validate_and_resolve_lora` 会读取或更新当前对象状态，建议继续查看该方法定义。
 613: 
 614:                 # Tokenize the request and send it to the scheduler
 615:                 if obj.is_single:
      # 注：分支判断：只有满足 `obj.is_single` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 616:                     tokenized_obj = await self._tokenize_one_request(obj)
      # 注：局部状态绑定：`tokenized_obj` 接住关键调用结果，后续代码会基于它继续装配组件或推进请求。
 617:                     state = self.rid_to_state[obj.rid]
      # 注：局部状态绑定：`state` 保存本阶段的中间结果，后续几行通常会立即消费它。
 618:                     if obj.return_prompt_token_ids:
      # 注：分支判断：只有满足 `obj.return_prompt_token_ids` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 619:                         state.prompt_token_ids = list(tokenized_obj.input_ids)
      # 注：对象状态写入：`state.prompt_token_ids` 修改 `state` 的可见状态，读源码时要继续追踪它在哪里被消费。
 620:                     self._send_one_request(tokenized_obj)
      # 注：关键调用：`self._send_one_request` - 把 tokenized request 通过 IPC 发给 scheduler。
 621:                     async for response in self._wait_one_response(obj, request):
      # 注：关键调用：`self._wait_one_response` - 等待 scheduler/detokenizer 回包，并按 streaming/non-streaming 产出 API 响应。
 622:                         yield response
      # 注：流式产出：把一个中间结果交给上游迭代器，函数状态会在下一次迭代时继续。
 623:                 else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 624:                     async for response in self._handle_batch_request(obj, request):
      # 注：成员函数调用：`self._handle_batch_request` 会读取或更新当前对象状态，建议继续查看该方法定义。
 625:                         yield response
      # 注：流式产出：把一个中间结果交给上游迭代器，函数状态会在下一次迭代时继续。
 626:         except Exception:
      # 注：异常处理分支：把失败转换成可控清理、缓存失败对象或用户可见错误。
 627:             # _init_req_state created a rid_to_state entry per (sub-)request up
 628:             # front. The normal remover is the scheduler-response path
 629:             # (_handle_batch_output), so a failure *before* a request reaches the
 630:             # scheduler -- e.g. input-length validation rejecting an over-context
 631:             # request -- would otherwise leak those entries forever. Drop any that
 632:             # are still pending; entries already removed on the normal completion
```

**读这段时抓住：**

- `auto_create_handle_loop()` 确保后台回包循环已经启动，否则请求发出去后没人消费 scheduler/detokenizer 输出。
- `normalize_batch_and_arguments()` 把单请求/批请求、多种参数别名规范化，这是后续代码能统一处理的前提。
- `model_update_lock.reader_lock` 保护权重/LoRA 更新期间的请求一致性；请求路径并不只处理文本。
- 单请求路径是 tokenize -> `_send_one_request` -> `_wait_one_response`；批请求走 `_handle_batch_request`，但仍会拆成内部 request state。
- 异常分支会清理 `rid_to_state`，这是防止早期校验失败导致内存状态泄漏的关键。


### `TokenizerManager.handle_loop`：为什么响应能回到原请求

```python
# python/sglang/srt/managers/tokenizer_manager.py:1824-1888
1824:     async def handle_loop(self):
      # 注：异步函数定义：`handle_loop` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
1825:         """The event loop that handles requests"""
1826:         while True:
      # 注：循环等待/轮询：注意退出条件，否则容易造成 busy wait 或阻塞。
1827:             with self.soft_watchdog.disable():
      # 注：成员函数调用：`self.soft_watchdog.disable` 会读取或更新当前对象状态，建议继续查看该方法定义。
1828:                 recv_obj = await self.recv_from_detokenizer.recv_pyobj()
      # 注：局部状态绑定：`recv_obj` 保存本阶段的中间结果，后续几行通常会立即消费它。
1829:             if isinstance(
      # 注：分支判断：只有满足 `isinstance(` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1830:                 recv_obj,
1831:                 (BatchStrOutput, BatchEmbeddingOutput, BatchTokenIDOutput),
1832:             ):
1833:                 await self._handle_batch_output(recv_obj)
      # 注：关键调用：`self._handle_batch_output` - 把 batch 输出拆回每个 rid，并整理 meta_info、文本、logprob、finish reason。
1834:             else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
1835:                 self._result_dispatcher(recv_obj)
      # 注：成员函数调用：`self._result_dispatcher` 会读取或更新当前对象状态，建议继续查看该方法定义。
1836:             self.last_receive_tstamp = real_time()
      # 注：成员变量写入：`self.last_receive_tstamp` 会留在对象生命周期中，后续方法可能依赖这份状态。
1837:             self.soft_watchdog.feed()
      # 注：成员函数调用：`self.soft_watchdog.feed` 会读取或更新当前对象状态，建议继续查看该方法定义。
1838: 
1839:     async def _handle_batch_output(
      # 注：异步函数定义：`_handle_batch_output` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
1840:         self,
1841:         recv_obj: Union[
      # 注：字段/变量声明：`recv_obj` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1842:             BatchStrOutput,
1843:             BatchEmbeddingOutput,
1844:             BatchTokenIDOutput,
1845:         ],
1846:     ):
1847:         pending_notify: dict[str, ReqState] = {}
      # 注：字段/变量声明：`pending_notify` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1848:         batch_notify_size = self.server_args.batch_notify_size
      # 注：局部状态绑定：`batch_notify_size` 保存请求或 batch 状态，后续调度/执行会继续改写它。
1849:         for i, rid in enumerate(recv_obj.rids):
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
1850:             state = self.rid_to_state.get(rid, None)
      # 注：局部状态绑定：`state` 保存本阶段的中间结果，后续几行通常会立即消费它。
1851:             if state is None:
      # 注：分支判断：只有满足 `state is None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1852:                 # Known race: /health_generate pops its rid as soon as ANY message bumps last_receive_tstamp.
1853:                 if rid.startswith(HEALTH_CHECK_RID_PREFIX):
      # 注：分支判断：只有满足 `rid.startswith(HEALTH_CHECK_RID_PREFIX)` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1854:                     continue
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
1855:                 logger.error(
      # 注：对象/库方法调用：`logger.error` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
1856:                     f"Received output for {rid=} but the state was deleted in TokenizerManager."
1857:                 )
1858:                 continue
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
1859: 
1860:             # Build meta_info and return value
1861:             meta_info = {
      # 注：局部状态绑定：`meta_info` 保存本阶段的中间结果，后续几行通常会立即消费它。
1862:                 "id": rid,
1863:                 "finish_reason": recv_obj.finished_reasons[i],
1864:                 "prompt_tokens": recv_obj.prompt_tokens[i],
1865:                 "weight_version": self.server_args.weight_version,
1866:                 "num_retractions": recv_obj.retraction_counts[i],
1867:             }
1868: 
1869:             if self.enable_metrics:
      # 注：分支判断：只有满足 `self.enable_metrics` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1870:                 if recv_obj.time_stats is not None:
      # 注：分支判断：只有满足 `recv_obj.time_stats is not None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1871:                     scheduler_time_stats = recv_obj.time_stats[i]
      # 注：局部状态绑定：`scheduler_time_stats` 保存本阶段的中间结果，后续几行通常会立即消费它。
1872:                     meta_info.update(scheduler_time_stats.convert_to_output_meta_info())
      # 注：对象/库方法调用：`meta_info.update` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
1873: 
1874:             if getattr(state.obj, "return_logprob", False):
      # 注：分支判断：只有满足 `getattr(state.obj, "return_logprob", False)` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1875:                 self.convert_logprob_style(
      # 注：成员函数调用：`self.convert_logprob_style` 会读取或更新当前对象状态，建议继续查看该方法定义。
1876:                     meta_info,
1877:                     state,
1878:                     state.obj.top_logprobs_num,
1879:                     state.obj.token_ids_logprob,
1880:                     state.obj.return_text_in_logprobs
1881:                     and not self.server_args.skip_tokenizer_init,
1882:                     recv_obj,
1883:                     i,
1884:                 )
1885: 
1886:             if not isinstance(recv_obj, BatchEmbeddingOutput):
      # 注：分支判断：只有满足 `not isinstance(recv_obj, BatchEmbeddingOutput)` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1887:                 meta_info.update(
      # 注：对象/库方法调用：`meta_info.update` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
1888:                     {
```

**读这段时抓住：**

- 它一直从 detokenizer/socket 读对象；Batch output 走 `_handle_batch_output`，控制类对象交给 dispatcher。
- `rid_to_state` 是回包路由表：scheduler 只知道 request id，API 层等待的是对应 state 的 event/output list。
- `meta_info` 在这里组装，所以许多用户可见统计信息并不是 model runner 直接返回的。
- streaming 的增量 offset、logprob 文本化、finish reason 都在这一层被整理成 API 输出。


In [ ]:
for path, names in [
    ("python/sglang/srt/managers/tokenizer_manager.py", {"generate_request", "_tokenize_one_request"}),
    ("python/sglang/srt/managers/scheduler.py", {"get_new_batch_prefill", "run_batch", "process_batch_result"}),
    ("python/sglang/srt/managers/schedule_batch.py", {"ScheduleBatch", "ForwardMode", "Req"}),
]:
    print("\n==", path)
    for row in find_defs(path, names=names):
        print(row)


### `engine.py` 初始化边界：manager 进程在这里被拼起来

```python
# python/sglang/srt/entrypoints/engine.py:117-175
 117: class SchedulerInitResult:
      # 注：类定义：`SchedulerInitResult` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
 118:     """Result from launching schedulers."""
 119: 
 120:     scheduler_infos: List[Dict[str, Any]]
      # 注：字段/变量声明：`scheduler_infos` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 121:     all_child_pids: List[int] = dataclasses.field(default_factory=list)
      # 注：字段/变量声明：`all_child_pids` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 122:     wait_for_ready: Callable[[], None] = lambda: None
      # 注：字段/变量声明：`wait_for_ready` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 123:     wait_for_completion: Callable[[], None] = lambda: None
      # 注：字段/变量声明：`wait_for_completion` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 124:     engine_info_bootstrap_server: Optional[Any] = None
      # 注：字段/变量声明：`engine_info_bootstrap_server` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 125: 
 126: 
 127: def init_tokenizer_manager(
      # 注：函数定义：`init_tokenizer_manager` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 128:     server_args: ServerArgs,
 129:     port_args: PortArgs,
 130:     TokenizerManagerClass: Optional[TokenizerManager] = None,
 131: ) -> Tuple[TokenizerManager, TemplateManager]:
 132:     # Launch tokenizer process
 133:     TokenizerManagerClass = TokenizerManagerClass or TokenizerManager
      # 注：局部状态绑定：`TokenizerManagerClass` 保存本阶段的中间结果，后续几行通常会立即消费它。
 134:     tokenizer_manager = TokenizerManagerClass(server_args, port_args)
      # 注：局部状态绑定：`tokenizer_manager` 保存本阶段的中间结果，后续几行通常会立即消费它。
 135: 
 136:     # Initialize templates
 137:     template_manager = TemplateManager()
      # 注：局部状态绑定：`template_manager` 保存本阶段的中间结果，后续几行通常会立即消费它。
 138:     template_manager.initialize_templates(
      # 注：对象/库方法调用：`template_manager.initialize_templates` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 139:         tokenizer_manager=tokenizer_manager,
 140:         model_path=server_args.model_path,
 141:         chat_template=server_args.chat_template,
 142:         completion_template=server_args.completion_template,
 143:     )
 144: 
 145:     # Resolve any remaining auto parsers using template manager's detection results
 146:     for attr, suggested, label in (
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
 147:         (
 148:             "reasoning_parser",
 149:             template_manager.suggested_reasoning_parser,
 150:             "reasoning parser",
 151:         ),
 152:         (
 153:             "tool_call_parser",
 154:             template_manager.suggested_tool_call_parser,
 155:             "tool-call parser",
 156:         ),
 157:     ):
 158:         if getattr(server_args, attr) != "auto":
      # 注：分支判断：根据启动参数 `getattr(server_args, attr) != "auto"` 选择服务拓扑、通信方式或功能开关。
 159:             continue
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
 160:         if suggested is not None:
      # 注：分支判断：只有满足 `suggested is not None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 161:             setattr(server_args, attr, suggested)
 162:             logger.info(
      # 注：对象/库方法调用：`logger.info` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 163:                 f"Auto-detected --{attr.replace('_', '-')} as '{suggested}' from chat template"
      # 注：对象/库方法调用：`attr.replace` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 164:             )
 165:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 166:             logger.warning(
      # 注：对象/库方法调用：`logger.warning` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 167:                 f"--{attr.replace('_', '-')}=auto specified but could not detect "
      # 注：对象/库方法调用：`attr.replace` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 168:                 f"{label} from chat template. Disabling {label}."
 169:             )
 170:             setattr(server_args, attr, None)
 171: 
 172:     return tokenizer_manager, template_manager
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 173: 
 174: 
 175: class Engine(EngineScoreMixin, EngineBase):
      # 注：类定义：`Engine` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
```

**读这段时抓住：**

- `SchedulerInitResult` 是 scheduler 初始化后回传给 HTTP/tokenizer 侧的能力摘要，例如 max input length、model info。
- `init_tokenizer_manager` 不是单纯构造 tokenizer，它会启动 scheduler/detokenizer 进程并建立 IPC 名称。
- 如果一个 Feature 需要跨 manager 传递新字段，通常要顺着这里确认哪个进程先知道它。


## 读 `Scheduler` 时不要迷路

`scheduler.py` 很长，因为它同时处理普通 decode、overlap schedule、grammar、speculative decoding、disaggregation、LoRA、profiling、metrics、weight update 等路径。建议按这几个函数读：

- `__init__`：所有子系统如何挂到 scheduler 上。
- `event_loop_normal` / `event_loop_overlap`：主循环形状。
- `get_new_batch_prefill`：prefill batch 进入 GPU 的入口。
- `run_batch`：从 `ScheduleBatch` 到 model worker forward。
- `process_batch_result`：采样、finish、streaming、cache 更新。


### `Scheduler.event_loop_normal`：普通调度主循环的形状

```python
# python/sglang/srt/managers/scheduler.py:1506-1535
1506:     def event_loop_normal(self):
      # 注：函数定义：`event_loop_normal` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
1507:         """A normal scheduler loop."""
1508:         while True:
      # 注：循环等待/轮询：注意退出条件，否则容易造成 busy wait 或阻塞。
1509:             if self.gracefully_exit:
      # 注：分支判断：只有满足 `self.gracefully_exit` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1510:                 break
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
1511: 
1512:             # Receive requests
1513:             recv_reqs = self.request_receiver.recv_requests()
      # 注：局部状态绑定：`recv_reqs` 保存请求或 batch 状态，后续调度/执行会继续改写它。
1514:             self.process_input_requests(recv_reqs)
      # 注：成员函数调用：`self.process_input_requests` 会读取或更新当前对象状态，建议继续查看该方法定义。
1515:             if self._engine_paused:
      # 注：分支判断：只有满足 `self._engine_paused` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1516:                 continue
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
1517: 
1518:             # Get the next batch to run
1519:             batch = self.get_next_batch_to_run()
      # 注：局部状态绑定：`batch` 保存请求或 batch 状态，后续调度/执行会继续改写它。
1520:             self.cur_batch = batch
      # 注：成员变量写入：`self.cur_batch` 会留在对象生命周期中，后续方法可能依赖这份状态。
1521: 
1522:             # Launch the current batch
1523:             if batch:
      # 注：分支判断：根据请求/batch 状态 `batch` 决定调度、执行或回包路径。
1524:                 result = self.run_batch(batch)
      # 注：局部状态绑定：`result` 接住关键调用结果，后续代码会基于它继续装配组件或推进请求。
1525:                 self.process_batch_result(batch, result)
      # 注：关键调用：`self.process_batch_result` - 把 forward/sampling 结果写回请求状态、发送输出并维护 cache。
1526:             else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
1527:                 # When the server is idle, do self-check and re-init some states.
1528:                 self.on_idle()
      # 注：成员函数调用：`self.on_idle` 会读取或更新当前对象状态，建议继续查看该方法定义。
1529: 
1530:             # Update last_batch
1531:             self.last_batch = batch
      # 注：成员变量写入：`self.last_batch` 会留在对象生命周期中，后续方法可能依赖这份状态。
1532:             if envs.SGLANG_ENABLE_STRICT_MEM_CHECK_DURING_BUSY.get():
      # 注：分支判断：只有满足 `envs.SGLANG_ENABLE_STRICT_MEM_CHECK_DURING_BUSY.get()` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1533:                 self.invariant_checker.self_check_during_busy()
      # 注：成员函数调用：`self.invariant_checker.self_check_during_busy` 会读取或更新当前对象状态，建议继续查看该方法定义。
1534: 
1535:     @DynamicGradMode()
      # 注：函数/库调用：`DynamicGradMode` 把当前阶段委托给外部 helper 或库实现。
```

**读这段时抓住：**

- 循环每轮先 `recv_requests`，再处理 grammar-ready、running batch、new prefill batch 等状态。
- `forward_ct` 和 watchdog 让 scheduler 可以暴露健康状态；调度循环本身也是生产系统对象。
- 普通循环和 overlap 循环分开，是因为 overlap schedule 要把 CPU 调度和 GPU forward 的依赖拆得更细。


### `Scheduler.event_loop_overlap`：重叠调度不是简单开线程

```python
# python/sglang/srt/managers/scheduler.py:1536-1565
1536:     def event_loop_overlap(self):
      # 注：函数定义：`event_loop_overlap` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
1537:         """A scheduler loop that overlaps the CPU processing and GPU computation."""
1538:         self.result_queue: Deque[
      # 注：字段/变量声明：`self.result_queue` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
1539:             Tuple[ScheduleBatch, Union[GenerationBatchResult, EmbeddingBatchResult]]
1540:         ] = deque()
      # 注：函数/库调用：`deque` 把当前阶段委托给外部 helper 或库实现。
1541: 
1542:         def pop_and_process():
      # 注：函数定义：`pop_and_process` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
1543:             # Process the results of the last batch
1544:             tmp_batch, tmp_result = self.result_queue.popleft()
      # 注：局部状态绑定：`tmp_batch, tmp_result` 保存请求或 batch 状态，后续调度/执行会继续改写它。
1545:             self.process_batch_result(tmp_batch, tmp_result)
      # 注：关键调用：`self.process_batch_result` - 把 forward/sampling 结果写回请求状态、发送输出并维护 cache。
1546: 
1547:         while True:
      # 注：循环等待/轮询：注意退出条件，否则容易造成 busy wait 或阻塞。
1548:             if self.gracefully_exit:
      # 注：分支判断：只有满足 `self.gracefully_exit` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1549:                 break
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
1550: 
1551:             # Receive requests
1552:             recv_reqs = self.request_receiver.recv_requests()
      # 注：局部状态绑定：`recv_reqs` 保存请求或 batch 状态，后续调度/执行会继续改写它。
1553:             self.process_input_requests(recv_reqs)
      # 注：成员函数调用：`self.process_input_requests` 会读取或更新当前对象状态，建议继续查看该方法定义。
1554:             if self._engine_paused:
      # 注：分支判断：只有满足 `self._engine_paused` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
1555:                 continue
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
1556: 
1557:             self._apply_war_barrier()
      # 注：成员函数调用：`self._apply_war_barrier` 会读取或更新当前对象状态，建议继续查看该方法定义。
1558: 
1559:             # Get the next batch to run
1560:             batch = self.get_next_batch_to_run()
      # 注：局部状态绑定：`batch` 保存请求或 batch 状态，后续调度/执行会继续改写它。
1561:             self.cur_batch = batch
      # 注：成员变量写入：`self.cur_batch` 会留在对象生命周期中，后续方法可能依赖这份状态。
1562:             disable_overlap_for_batch = self.is_disable_overlap_for_batch(batch)
      # 注：局部状态绑定：`disable_overlap_for_batch` 保存请求或 batch 状态，后续调度/执行会继续改写它。
1563: 
1564:             # If we do not need to overlap the current batch with the last batch,
1565:             # we can process the last batch immediately.
```

**读这段时抓住：**

- overlap 路径有专门的 `result_queue`，上一轮 GPU 结果可能在下一轮 CPU 调度时才回收。
- 许多 Feature 要分别验证 normal/overlap 两条路径，例如 speculative、grammar、abort、pause generation。
- 读 scheduler bug 时，要先确认当前服务是否启用了 overlap，否则你可能在读错主循环。


In [ ]:
show_lines("python/sglang/srt/managers/scheduler.py", 1500, 1565)


## 小练习：定位一次 `/generate` 的关键函数

下面的 cell 不启动服务，只用 AST 找出关键定义。读源码时可以把这些行号作为入口点。


In [ ]:
targets = {
    "python/sglang/srt/entrypoints/http_server.py": {"generate_request", "launch_server"},
    "python/sglang/srt/managers/tokenizer_manager.py": {"generate_request"},
    "python/sglang/srt/managers/scheduler.py": {"get_new_batch_prefill", "run_batch", "process_batch_result"},
}
for path, names in targets.items():
    print("\n" + path)
    for lineno, kind, name in find_defs(path, names):
        print(f"  {name}: line {lineno}")
